# Orbit ODE Notebook

## 1) Imports and Global Constants

In [23]:
using LinearAlgebra
using Statistics
using DifferentialEquations
using StaticArrays
using MeshCat
using GeometryBasics
using CoordinateTransformations
using Colors

const MU = 398600.4418      # km^3/s^2 , earths gravitation parameter
const RE = 6378.1363        # km, earth radius
const J2 = 1.08262668e-3    # earth oblateness constant, important to us, sunsync


0.00108262668

## 2) Dynamics Functions

In [24]:
function drag_accel_km(r_km, v_kms)
    r_m = 1000.0 .* r_km
    v_ms = 1000.0 .* v_kms

    rn_m = norm(r_m)
    h = rn_m - RE*1000.0

    rho = rho0 * exp(-(h - h0)/H)

    w = SVector(0.0, 0.0, wE)
    v_atm = cross(w, r_m)
    v_rel = v_ms - v_atm
    vrel = norm(v_rel)

    aD_ms2 = -0.5 * rho * Bc * vrel * v_rel
    return aD_ms2 ./ 1000.0   # km/s^2
end

# p is a NamedTuple with field :drag (Bool)
# p = (drag=false,)  →  J2-only (for SSO Ωdot verification)
# p = (drag=true,)   →  J2 + atmospheric drag (full model)
function eci_ode!(dx, x, p, t)
    r = @view x[1:3]
    v = @view x[4:6]
    rn = norm(r)

    # two-body
    a2b = -MU .* r ./ rn^3

    xk, yk, zk = r
    z2 = zk*zk
    r2 = rn*rn
    fac = 1.5 * J2 * MU * RE^2 / rn^5
    aJ2 = [fac*xk*(5*z2/r2 - 1),
           fac*yk*(5*z2/r2 - 1),
           fac*zk*(5*z2/r2 - 3)]

    aDrag = p.drag ? drag_accel_km(r, v) : zeros(3)

    a = a2b + aJ2 + aDrag
    dx[1:3] .= v
    dx[4:6] .= a
end


eci_ode! (generic function with 1 method)

## 3) Earth Rotation and Drag Parameters

In [25]:
const wE = 7.2921159e-5     #rad/s Earth rotation

const rho0 = 3.614e-13        # kg/m^3 at h0=700 km (rough)
const h0 = 700e3            # m
const H  = 88.667e3         # m scale height (rough)

const Cd = 2.2
const A  = 0.03          # m^2 (example)
const m  = 12.0          # kg  (example)
const Bc = Cd*A/m        # m^2/kg


0.0055000000000000005

## 4) Sun-Synchronous Orbit Setup

The Starling mission flies a **Sun-synchronous orbit (SSO) at ~480 km altitude**. The derivation below follows standard astrodynamics (Vallado / SMAD).

**J2 nodal precession rate** (secular, averaged over one orbit for a near-circular orbit):

$$\dot\Omega = -\frac{3}{2} J_2 n \left(\frac{R_E}{p}\right)^2 \cos i, \qquad
  n = \sqrt{\frac{\mu}{a^3}}, \quad p = a(1-e^2)$$

For SSO we require $\dot\Omega = +0.985647\,^\circ/\text{day}$ (Earth's mean orbital rate around the Sun),
forcing the orbital plane to track the Sun year-round.
Because the nodal regression is negative for prograde orbits, SSO requires a **retrograde inclination** ($i > 90^\circ$):

$$\cos i = \frac{\dot\Omega}{-\tfrac{3}{2} J_2 n (R_E/p)^2}$$

*Note:* ECI frame conventions follow Lecture 1 §3 (attitude = relative rotation between body frame $\mathcal{B}$ and ECI frame $\mathcal{N}$).
Rotation-matrix kinematics ($\dot Q = Q\,{}^B\hat\omega$) follow Lecture 2 eq. (1).


In [26]:
# --- SSO inclination for Starling at 480 km (matches report) ---

const OMEGA_DOT_TARGET_DEG_DAY = 0.985647          # deg/day  (Earth mean orbital motion)
const OMEGA_DOT_TARGET = OMEGA_DOT_TARGET_DEG_DAY * (π/180) / 86400  # rad/s

const alt_sso = 480.0                              # km  — locked to report value
const a_sso   = RE + alt_sso                       # semi-major axis, km
const n_sso   = sqrt(MU / a_sso^3)                 # mean motion, rad/s
const p_sso   = a_sso                              # e ≈ 0 → p = a
const T_orbit = 2π / n_sso                         # orbital period, seconds

# Solve for SSO inclination
const cos_i_sso = OMEGA_DOT_TARGET / (-1.5 * J2 * n_sso * (RE/p_sso)^2)
const i_sso     = acos(cos_i_sso)                  # rad, retrograde (i > 90°)

println("SSO parameters for altitude = $alt_sso km:")
println("  a        = $(round(a_sso,    digits=4)) km")
println("  n        = $(round(n_sso*1e3,digits=6)) ×10⁻³ rad/s")
println("  T_orbit  = $(round(T_orbit,  digits=2)) s  ($(round(T_orbit/60, digits=2)) min)")
println("  Target Ω̇ = $OMEGA_DOT_TARGET_DEG_DAY deg/day")
println("  cos(i)   = $(round(cos_i_sso, digits=6))")
println("  i_SSO    = $(round(rad2deg(i_sso), digits=3))°  (retrograde, > 90°)")


SSO parameters for altitude = 480.0 km:
  a        = 6858.1363 km
  n        = 1.111629 ×10⁻³ rad/s
  T_orbit  = 5652.23 s  (94.2 min)
  Target Ω̇ = 0.985647 deg/day
  cos(i)   = -0.12752
  i_SSO    = 97.326°  (retrograde, > 90°)


## 5) Orbital Mechanics Helpers


In [27]:
# COE → ECI  (all angles in radians, distances in km)
# Standard PQW-to-ECI DCM: Q = R3(Ω) * R1(i) * R3(ω)
function coe_to_eci(a_km, e, inc, Ω, ω, ν)
    p  = a_km * (1 - e^2)
    rn = p / (1 + e*cos(ν))

    r_pqw = rn        .* [cos(ν),          sin(ν),         0.0]
    v_pqw = sqrt(MU/p) .* [-sin(ν),  e + cos(ν),           0.0]

    cΩ, sΩ = cos(Ω), sin(Ω)
    ci, si = cos(inc), sin(inc)
    cω, sω = cos(ω), sin(ω)

    # DCM columns:  Q[:,j] = jth perifocal basis vector expressed in ECI
    Q = [cΩ*cω - sΩ*ci*sω   -cΩ*sω - sΩ*ci*cω   sΩ*si;
         sΩ*cω + cΩ*ci*sω   -sΩ*sω + cΩ*ci*cω  -cΩ*si;
         si*sω                si*cω               ci   ]

    return Q*r_pqw, Q*v_pqw
end

# RAAN from ECI state — node vector N = k̂ × h, Ω = atan2(N_y, N_x)
function compute_raan(r, v)
    h  = cross(r, v)
    N  = [-h[2], h[1], 0.0]          # k̂ × h
    Nn = sqrt(N[1]^2 + N[2]^2)
    Nn < 1e-10 && return 0.0          # equatorial edge case
    Ω  = atan(N[2], N[1])
    return Ω < 0 ? Ω + 2π : Ω
end

# Semi-major axis from vis-viva
function compute_sma(r, v)
    ε = dot(v,v)/2 - MU/norm(r)
    return -MU / (2ε)
end

# Inclination from angular momentum direction
function compute_inc(r, v)
    h = cross(r, v)
    return acos(clamp(h[3]/norm(h), -1.0, 1.0))
end

println("coe_to_eci, compute_raan, compute_sma, compute_inc defined")


coe_to_eci, compute_raan, compute_sma, compute_inc defined


## 6) Initial Conditions — SSO at 480 km

In [28]:
# COE initial conditions: ascending node on +X axis, ν=0 (perigee = ascending node for e=0)
# Ω=0, ω=0, ν=0  →  r₀ = [a, 0, 0],  v₀ = [0, vc·cos(i), vc·sin(i)]
r0, v0 = coe_to_eci(a_sso, 0.0, i_sso, 0.0, 0.0, 0.0)
x0 = vcat(r0, v0)

println("Initial ECI state:")
println("  r₀ = ", round.(r0, digits=4), " km")
println("  v₀ = ", round.(v0, digits=6), " km/s")
println("  |r₀| = $(round(norm(r0), digits=4)) km")
println("  |v₀| = $(round(norm(v0), digits=6)) km/s")
println("  a₀  = $(round(compute_sma(r0, v0), digits=4)) km  (check = $a_sso km)")
println("  i₀  = $(round(rad2deg(compute_inc(r0, v0)), digits=4))°  (check = $(round(rad2deg(i_sso), digits=4))°)")
println("  Ω₀  = $(round(rad2deg(compute_raan(r0, v0)), digits=4))°")


Initial ECI state:
  r₀ = [6858.1363, 0.0, 0.0] km
  v₀ = [0.0, -0.972178, 7.56146] km/s
  |r₀| = 6858.1363 km
  |v₀| = 7.623701 km/s
  a₀  = 6858.1363 km  (check = 6858.1363 km)
  i₀  = 97.3263°  (check = 97.3263°)
  Ω₀  = 0.0°


In [29]:
# Case 1: J2-only  (no drag)
# SSO nodal-precession formula assumes conservative J2 only;
# drag breaks the condition, so verify with drag OFF first.
tspan = (0.0, 10.0 * T_orbit)     # 10 orbits ≈ 15.7 hours → clear Ωdot slope

prob_j2 = ODEProblem(eci_ode!, x0, tspan, (drag=false,))
sol_j2  = solve(prob_j2, Vern9();
    reltol=1e-10, abstol=1e-12,
    saveat=30.0, dtmax=60.0)

println("Case 1 (J2-only): $(length(sol_j2.u)) steps,  tspan = $(round.(tspan ./ 3600, digits=2)) hr")
println("  a_final = $(round(compute_sma(sol_j2.u[end][1:3], sol_j2.u[end][4:6]), digits=4)) km  (should match a_sso)")
println("  i_final = $(round(rad2deg(compute_inc(sol_j2.u[end][1:3], sol_j2.u[end][4:6])), digits=4))°")


Case 1 (J2-only): 1886 steps,  tspan = (0.0, 15.7) hr
  a_final = 6858.0936 km  (should match a_sso)
  i_final = 97.3264°


In [30]:
# Case 2: J2 + atmospheric drag  (full model, used for animation)
prob = ODEProblem(eci_ode!, x0, tspan, (drag=true,))
sol  = solve(prob, Vern9();
    reltol=1e-10, abstol=1e-12,
    saveat=30.0, dtmax=60.0)

a_end = compute_sma(sol.u[end][1:3], sol.u[end][4:6])
println("Case 2 (J2+drag): $(length(sol.u)) steps")
println("  a_final = $(round(a_end, digits=4)) km  (Δa = $(round(a_end - a_sso, digits=4)) km)")
println("  i_final = $(round(rad2deg(compute_inc(sol.u[end][1:3], sol.u[end][4:6])), digits=4))°")
println("  Note: drag causes slow SMA decay; true SSO will drift as altitude drops.")


Case 2 (J2+drag): 1886 steps
  a_final = 6858.0173 km  (Δa = -0.119 km)
  i_final = 97.3263°
  Note: drag causes slow SMA decay; true SSO will drift as altitude drops.


In [31]:
# Verify RAAN precession rate using Case 1 (J2-only)
Ω_hist = [compute_raan(sol_j2.u[k][1:3], sol_j2.u[k][4:6]) for k in 1:length(sol_j2.u)]

# Unwrap 2π phase jumps
Ω_uw = copy(Ω_hist)
for k in 2:length(Ω_uw)
    dΩ = Ω_uw[k] - Ω_uw[k-1]
    if dΩ >  π;  Ω_uw[k] -= 2π; end
    if dΩ < -π;  Ω_uw[k] += 2π; end
end

# Least-squares linear fit for slope (rad/s)
t_arr   = sol_j2.t
t_c     = t_arr .- mean(t_arr)
Ω_c     = Ω_uw  .- mean(Ω_uw)
slope_rad_s  = sum(t_c .* Ω_c) / sum(t_c .^ 2)
slope_deg_day = slope_rad_s * (180/π) * 86400

println("=== RAAN Precession Verification (Case 1: J2-only) ===")
println("  Simulated  Ω̇ = $(round(slope_deg_day,          digits=6)) deg/day")
println("  Target     Ω̇ = $OMEGA_DOT_TARGET_DEG_DAY deg/day")
println("  Error      Δ = $(round(abs(slope_deg_day - OMEGA_DOT_TARGET_DEG_DAY), sigdigits=3)) deg/day")
println()
println("=== Drag Effect on SMA (Case 2 vs Case 1) ===")
a_j2_end   = compute_sma(sol_j2.u[end][1:3], sol_j2.u[end][4:6])
a_drag_end = compute_sma(sol.u[end][1:3],    sol.u[end][4:6])
println("  J2-only  a_final = $(round(a_j2_end,   digits=4)) km  (conserved ✓)")
println("  J2+drag  a_final = $(round(a_drag_end, digits=4)) km  (Δa = $(round(a_drag_end - a_sso, digits=4)) km)")


=== RAAN Precession Verification (Case 1: J2-only) ===
  Simulated  Ω̇ = 0.991003 deg/day
  Target     Ω̇ = 0.985647 deg/day
  Error      Δ = 0.00536 deg/day

=== Drag Effect on SMA (Case 2 vs Case 1) ===
  J2-only  a_final = 6858.0936 km  (conserved ✓)
  J2+drag  a_final = 6858.0173 km  (Δa = -0.119 km)


## 7) Plots for Report

In [ ]:
using PythonPlot

# --- Plot 1: 3D orbit trajectory (J2+drag, 10 orbits) ---
R2 = hcat([u[1:3] for u in sol.u]...)

fig = figure(figsize=(7, 6))
ax  = fig.add_subplot(111, projection="3d")

# Earth sphere (wireframe)
u_sph = range(0, 2π, length=40)
v_sph = range(0, π,  length=20)
Xs = [RE * cos(u)*sin(v) for u in u_sph, v in v_sph]
Ys = [RE * sin(u)*sin(v) for u in u_sph, v in v_sph]
Zs = [RE * cos(v)        for u in u_sph, v in v_sph]
ax.plot_wireframe(Xs, Ys, Zs, color="cornflowerblue", alpha=0.15, linewidth=0.4)

# Orbit trajectory
ax.plot(R2[1,:], R2[2,:], R2[3,:], color="tomato", linewidth=1.0, label="Orbit (J2+drag)")

# Mark start
ax.scatter([R2[1,1]], [R2[2,1]], [R2[3,1]], color="green", s=30, zorder=5, label="Epoch (t=0)")

ax.set_xlabel("X (km)"); ax.set_ylabel("Y (km)"); ax.set_zlabel("Z (km)")
ax.set_title("Starling SSO — ECI trajectory, 10 orbits\nalt = 480 km,  i = 97.326°")
ax.legend(loc="upper left", fontsize=8)
ax.set_box_aspect([1,1,1])
tight_layout()
savefig("figs/orbit_trajectory.png", dpi=150, bbox_inches="tight")
println("Saved figs/orbit_trajectory.png")


In [ ]:
# --- Plot 2: RAAN vs. time  (J2-only, with target slope overlaid) ---
t_hr   = sol_j2.t ./ 3600                          # seconds → hours
Ω_deg  = rad2deg.(Ω_uw)                             # unwrapped RAAN in degrees

# Target line from Ω₀=0 at slope 0.985647 deg/day = 0.985647/24 deg/hr
target_slope_deg_hr = OMEGA_DOT_TARGET_DEG_DAY / 24
Ω_target = target_slope_deg_hr .* t_hr

fig2, ax2 = subplots(figsize=(7, 4))
ax2.plot(t_hr, Ω_deg,    color="tomato",      linewidth=1.5, label="Simulated Ω (J2-only)")
ax2.plot(t_hr, Ω_target, color="steelblue",   linewidth=1.2,
         linestyle="--", label="Target slope  $(OMEGA_DOT_TARGET_DEG_DAY) °/day")

ax2.set_xlabel("Time (hours)")
ax2.set_ylabel("RAAN  Ω  (degrees)")
ax2.set_title("RAAN Precession — SSO Verification\nSimulated $(round(slope_deg_day, digits=4)) °/day  vs  target $(OMEGA_DOT_TARGET_DEG_DAY) °/day")
ax2.legend()
ax2.grid(true, alpha=0.3)
tight_layout()
savefig("figs/raan_precession.png", dpi=150, bbox_inches="tight")
println("Saved figs/raan_precession.png")


## 5) Looping Orbit Animation (MeshCat)

In [35]:
R = hcat([u[1:3] for u in sol.u]...)
N = size(R, 2)

# Normalize to Earth-radius display units:
#   Earth sphere → radius 1.0, orbit → ~1.09 units from center
#   Without this the camera (at ~3 units) is inside the 6378-km sphere
s = 1.0 / RE

max_frames = 1200
step = max(1, cld(N, max_frames))
idx = collect(1:step:N)
xv = Float32.(R[1, idx] .* s)
yv = Float32.(R[2, idx] .* s)
zv = Float32.(R[3, idx] .* s)
M  = length(xv)

vis = Visualizer()
open(vis)

# Earth: unit sphere with blue material
setobject!(vis[:earth],
    Sphere(Point(0.0, 0.0, 0.0), 1.0),
    MeshPhongMaterial(color=RGBA(0.2, 0.5, 1.0, 0.9)))

# Satellite: small red box (~0.05 RE ≈ 320 km display size — visible from default camera)
sat_d = 0.05
setobject!(vis[:sat],
    Rect(Vec(-sat_d/2, -sat_d/2, -sat_d/2), Vec(sat_d, sat_d, sat_d)),
    MeshPhongMaterial(color=colorant"red"))

anim = Animation(vis)
for t = 1:M
    atframe(anim, t) do
        settransform!(vis[:sat], Translation(xv[t], yv[t], zv[t]))
    end
end

setanimation!(vis, anim; play=true, repetitions=1000000)


[ Info: Listening on: 127.0.0.1:8708, thread id: 1
┌ Info: MeshCat server started. You can open the visualizer by visiting the following URL in your browser:
└ http://127.0.0.1:8708


In [36]:
#okay so what do we have
#Earth Gravity
#Earth oblateness J2
#ECI Coordinates
#acc alt

#for later
#drag
#solar rad pressure
#third body consideration
#input or maneuvers